In [8]:
from datasets import load_dataset
dataset = load_dataset("nyu-mll/glue","qqp",split="train[240000:290000]")
print(f"Loaded {len(dataset_subset)} records.")
print(dataset_subset[0])

Loaded 50000 records.
{'question1': 'Are cruise missiles fake?', 'question2': 'Are missiles fake?', 'label': 0, 'idx': 240000}


In [9]:
questions= list(dataset['question1']) + list(dataset['question2'])

In [10]:
len(questions)

100000

CREATING INDEX

In [ ]:
index_name="quora-embeddings"

In [14]:
!pip install openai pinecone transformers datasets sentence-transformers

In [15]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
os.environ['HUGGINGFACE_API_KEY'] = userdata.get('HUGGINGFACE_API_KEY')

In [16]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
pc=Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
index_name ="quora-embeddings"
if not pc.has_index(index_name):
 pc.create_index(
 name=index_name,
 vector_type="dense",
 dimension=384,
 metric="cosine",
 spec=ServerlessSpec(
 cloud="aws",
 region="us-east-1"
 ),
 deletion_protection="disabled",
 )

In [18]:
pc.list_indexes()

IndexList([<name='chirag-vectors', ready=True>, <name='quora-embeddings', ready=True>])

In [26]:
import math
from sentence_transformers import SentenceTransformer

# Select the first 10,000 questions
texts = questions[:10000]

# Connect to the Pinecone index
index = pc.Index("quora-embeddings")

# Load the embedding model
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def upsert_in_batches(texts, batch_size=200):
    total = len(texts)
    num_batches = math.ceil(total / batch_size)

    print(f"Total records: {total}")
    print(f"Batch size: {batch_size}")
    print(f"Total batches: {num_batches}")

    for batch_num in range(num_batches):
        start = batch_num * batch_size
        end = min(start + batch_size, total)

        batch_texts = texts[start:end]

        # Generate embeddings for the current batch
        embeddings = model.encode(batch_texts)

        # Prepare vectors for Pinecone
        vectors = []

        for i, (text, embedding) in enumerate(
            zip(batch_texts, embeddings)
        ):
            vector = {
                "id": str(start + i),
                "values": embedding.tolist(),
                "metadata": {
                    "text": text
                }
            }

            vectors.append(vector)

        # Upsert the current batch into Pinecone
        index.upsert(vectors=vectors)

        print(
            f"Batch {batch_num + 1}/{num_batches} upserted"
        )

    print("All batches upserted successfully!")


# Run the batch upsert
upsert_in_batches(texts, batch_size=200)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Total records: 10000
Batch size: 200
Total batches: 50
Batch 1/50 upserted
Batch 2/50 upserted
Batch 3/50 upserted
Batch 4/50 upserted
Batch 5/50 upserted
Batch 6/50 upserted
Batch 7/50 upserted
Batch 8/50 upserted
Batch 9/50 upserted
Batch 10/50 upserted
Batch 11/50 upserted
Batch 12/50 upserted
Batch 13/50 upserted
Batch 14/50 upserted
Batch 15/50 upserted
Batch 16/50 upserted
Batch 17/50 upserted
Batch 18/50 upserted
Batch 19/50 upserted
Batch 20/50 upserted
Batch 21/50 upserted
Batch 22/50 upserted
Batch 23/50 upserted
Batch 24/50 upserted
Batch 25/50 upserted
Batch 26/50 upserted
Batch 27/50 upserted
Batch 28/50 upserted
Batch 29/50 upserted
Batch 30/50 upserted
Batch 31/50 upserted
Batch 32/50 upserted
Batch 33/50 upserted
Batch 34/50 upserted
Batch 35/50 upserted
Batch 36/50 upserted
Batch 37/50 upserted
Batch 38/50 upserted
Batch 39/50 upserted
Batch 40/50 upserted
Batch 41/50 upserted
Batch 42/50 upserted
Batch 43/50 upserted
Batch 44/50 upserted
Batch 45/50 upserted
Batch 46/

In [27]:
query = "which city has the highest population in the world?"
q_embeddings = model.encode(query).tolist( )
q_embeddings

[0.13341057300567627,
 -0.023074602708220482,
 -0.05944643169641495,
 0.043846968561410904,
 -0.06582780927419662,
 -0.03988824784755707,
 0.008183957077562809,
 0.03740554302930832,
 -0.05933888629078865,
 0.041530631482601166,
 0.0662338137626648,
 -0.08682147413492203,
 0.008207847364246845,
 0.04380276799201965,
 0.00019493709260132164,
 -0.01594242826104164,
 0.019748535007238388,
 -0.04541019722819328,
 0.06308892369270325,
 -0.06579317152500153,
 -0.021352222189307213,
 -0.05284406617283821,
 0.10262908786535263,
 0.03393443673849106,
 0.011895764619112015,
 0.0004714914539363235,
 -0.01892256550490856,
 0.0691978856921196,
 -0.011790876276791096,
 0.004062442108988762,
 0.00018102963804267347,
 0.0513603650033474,
 0.13404084742069244,
 0.0012998334132134914,
 0.01660400815308094,
 0.011573241092264652,
 -0.010894499719142914,
 -0.03396617993712425,
 0.04109124466776848,
 0.04152518883347511,
 0.025995055213570595,
 -0.026188256219029427,
 -0.009932654909789562,
 -0.04452628269

In [34]:
index = pc.Index("quora-embeddings")
response = index.query(
 namespace="__default__",
 vector=q_embeddings,
 top_k=3,
 include_metadata=True,
 include_values=False
)
for match in response.matches:
 print(f"{match.id}, {match.get('metadata')['text']}, {match.score}")

1714, Where is the most beautiful city in the world?, 0.696199954
6556, Which is the best city in the world to live in?, 0.695678592
683, Which is the best city in the world to travel?, 0.658938289


In [37]:
contexts = [
    x['metadata']['text'] for x in response['matches']
]
contexts

['Where is the most beautiful city in the world?',
 'Which is the best city in the world to live in?',
 'Which is the best city in the world to travel?']

In [ ]:
###Augumentation

In [38]:
prompt = f"""
Answer the question based on the context below.\n\n
Context:\n
{"\n\n---\n\n".join(contexts)}
\n\nQuestion: {query}\nAnswer: """
print(prompt)

 
Answer the question based on the context below.

 
Context:

Where is the most beautiful city in the world?

---

Which is the best city in the world to live in?

---

Which is the best city in the world to travel?


Question: which city has the highest population in the world?
Answer: 


In [ ]:
##GENERATION

In [39]:
from openai import OpenAI

client = OpenAI()

messages = [
    {
        "role": "system",
        "content": "You are a helpful and precise assistant."
    },
    {
        "role": "user",
        "content": prompt
    }
]

print("Input:", messages)

res = client.chat.completions.create(
    model="gpt-4o",  # Model for fast inference
    messages=messages,
    temperature=0,  # Deterministic output for factual tasks
    max_tokens=636,  # Maximum number of generated tokens
    top_p=1,  # Consider the full probability distribution
    frequency_penalty=0,  # No penalty for repeated phrases
    presence_penalty=0,  # No penalty for introducing new topics
    stop=None  # No custom stop sequence
)

print("-" * 80)
print(res.choices[0].message.content)

Input: [{'role': 'system', 'content': 'You are a helpful and precise assistant.'}, {'role': 'user', 'content': ' \nAnswer the question based on the context below.\n\n \nContext:\n\nWhere is the most beautiful city in the world?\n\n---\n\nWhich is the best city in the world to live in?\n\n---\n\nWhich is the best city in the world to travel?\n\n\nQuestion: which city has the highest population in the world?\nAnswer: '}]
--------------------------------------------------------------------------------
As of the latest data, Tokyo, Japan, is often cited as the city with the highest population in the world.
